---



Name: Sharmishtha V Bhimanpalli

SRN:PES2UG23CS914

# Part 3a: Chain of Thought (CoT)


# Concept: Latent Reasoning

In [1]:
# Setup
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b (Small/Fast) to demonstrate logic failures
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.0/342.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.6/212.6 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
Note: you may need to restart the kernel to use updated packages.


Enter your Groq API Key:  ········


## Experiment: A Tricky Math Problem
#### 1. Direct prompt

**Problem:**
"Lisa has 10 stickers. She buys 2 sheets of stickers, each sheet has 6 stickers. She uses 5 stickers for decoration and gives half of the remaining stickers to her brother. How many stickers does she have left?"

In [7]:
question = "Lisa has 10 stickers. She buys 2 sheets of stickers, each sheet has 6 stickers. She uses 5 stickers for decoration and gives half of the remaining stickers to her brother. How many stickers does she have left?"

# 1. Standard Prompt (Direct Answer)
prompt_standard = f"Answer this question: {question}"
print("--- STANDARD (Llama3.1-8b) ---")
print(llm.invoke(prompt_standard).content)

--- STANDARD (Llama3.1-8b) ---
To determine how many stickers Lisa has left, we need to follow the steps she takes:

1. Lisa starts with 10 stickers.
2. She buys 2 sheets of stickers, each with 6 stickers. This means she gets 2 x 6 = 12 stickers.
3. She adds these new stickers to her original 10 stickers: 10 + 12 = 22 stickers.
4. She uses 5 stickers for decoration: 22 - 5 = 17 stickers left.
5. She gives half of the remaining stickers to her brother. Half of 17 is 8.5, but since you can't give a fraction of a sticker, we can assume she gives 8 stickers to her brother.
6. After giving 8 stickers to her brother, Lisa has 17 - 8 = 9 stickers left.

So, Lisa has 9 stickers left.


In [8]:
# 2. CoT Prompt (Magic Phrase)
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- Chain of Thought (Llama3.1-8b) ---")
print(llm.invoke(prompt_cot).content)

--- Chain of Thought (Llama3.1-8b) ---
Let's break down the problem step by step:

1. Lisa starts with 10 stickers. 

2. She buys 2 sheets of stickers, each sheet has 6 stickers. So she gets 2 x 6 = 12 stickers. 
   Adding this to her initial stickers, she now has 10 + 12 = 22 stickers.

3. Lisa uses 5 stickers for decoration. Now she has 22 - 5 = 17 stickers left.

4. She gives half of the remaining stickers to her brother. Half of 17 is 17 / 2 = 8.5. Since you can't give a fraction of a sticker, Lisa will round down to 8 stickers for her brother.

5. Now Lisa has 17 - 8 = 9 stickers left.


#### Analysis

Look at the output. By explicitly breaking it down:
1.  "Lisa starts with 10."
2.  "2 sheets * 6 stickers = 12 stickers."
3.  12 + 10 = 22
4.  uses 5 for decoration.
5.  "22 - 5 = 17."
6.  give half to her brother.
7.  17 / 2 = 8.5 ~ 8.
8.  lisa is left with
9.  17 - 8 = 9 stickers.

The model effectively "debugs" its own logic by generating the intermediate steps.

---



# Part 3b: Tree of Thoughts (ToT) & Graph of Thoughts (GoT)


In [9]:
# Setup
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7) # Creativity needed

Note: you may need to restart the kernel to use updated packages.


## 2. Tree of Thoughts (ToT)


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "“How can a college student improve focus while studying without caffeine?"

# Step 1: The Branch Generator
prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Suggest one practical and unique strategy. Avoid caffeine-related solutions. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# Step 2: The Judge
prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'
    
    1: {sol1}
    2: {sol2}
    3: {sol3}
    
    Act as an Academic Productivity Coach. 
    Pick the most sustainable and healthy solution and explain why.
    """
)

# Chain: Input -> Branches -> Judge -> Output
tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]}) 
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
As an Academic Productivity Coach, I highly recommend Solution 3: **Use the "Pomodoro Technique with a Twist: Natural Light Breaks"**. This solution offers a sustainable and healthy approach to improving focus while studying without relying on caffeine.

Here's why I prefer this solution:

1. **Natural and effortless**: Unlike Solution 1, which requires creating a virtual forest, or Solution 2, which involves sensory stimulation, Solution 3 leverages the natural environment to improve focus. This approach is effortless and doesn't require any additional equipment or setup.
2. **Physical and mental benefits**: Exposure to natural light has been shown to increase alertness, energy, and focus, which aligns perfectly with the goals of improving focus. Additionally, taking breaks in natural light can help reduce eye strain and fatigue, promoting overall well-being.
3. **Flexibility and adaptability**: Solution 3 can be adapted to various study environme

### **Analysis**

The Tree-of-Thought approach generated multiple possible strategies and then evaluated them through a judging step. The model selected the “Pomodoro Technique with Natural Light Breaks” as the most sustainable solution because it combines structured study intervals with environmental benefits like natural light and fresh air. Compared to the other solutions, it was judged more practical, healthier, and easier to maintain long-term, demonstrating how structured branching plus evaluation can guide the model toward more reasoned and balanced conclusions.


## 3. Graph of Thoughts (GoT)


In [12]:
# 1. The Generator (Divergence)
prompt_draft = ChatPromptTemplate.from_template(
    "Suggest a 1–2 sentence strategy about: {topic}. Perspective: {genre}."
)

drafts = RunnableParallel(
    draft_scientific=prompt_draft.partial(genre="Scientific") | llm | StrOutputParser(),
    draft_psychological=prompt_draft.partial(genre="Psychological") | llm | StrOutputParser(),
    draft_lifestyle=prompt_draft.partial(genre="Lifestyle") | llm | StrOutputParser(),
)

# 2. The Aggregator (Convergence)
prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three strategies for '{topic}':

    1. Scientific: {draft_scientific}
    2. Psychological: {draft_psychological}
    3. Lifestyle: {draft_lifestyle}

    Combine the best ideas into ONE practical, balanced strategy.
    Write one short paragraph.
    """
)

# 3. The Chain
got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]}) 
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
got_chain.invoke("Improving focus while studying")

--- Graph of Thoughts (GoT) Result ---


"To improve focus while studying, combine the best elements of the three strategies into a balanced routine. Start by establishing a consistent morning routine that includes physical activity, healthy nutrition, and dedicated study time before distractions arise. Throughout the day, use the Pomodoro Technique to work in focused 25-minute increments, separated by 5-minute breaks, allowing your brain to rest and recharge. Also, incorporate 90-120 minute study blocks, separated by 10-15 minute breaks, to align with your body's natural ultradian rhythms and optimize cognitive processing and retention. By prioritizing a well-rounded morning routine and combining focused work sessions with regular breaks, you'll be better equipped to maintain your concentration and achieve your study goals."

### **Analysis**

The Graph-of-Thought approach generated multiple strategies from different perspectives and then combined them into a single cohesive solution. The final output integrates scientific, psychological, and lifestyle elements—such as structured study intervals, healthy routines, and regular breaks—into one balanced strategy. This demonstrates how Graph-of-Thought prompting enables synthesis of diverse ideas to produce a more comprehensive and practical recommendation.



## 4. Summary & Comparison Table

- Chain-of-Thought (CoT) showed step-by-step linear reasoning to solve a math problem.
- Tree-of-Thought (ToT) explored multiple possible solutions in parallel and then evaluated them to select the most sustainable option.
- Graph-of-Thought (GoT) went a step further by synthesizing ideas from multiple perspectives into one integrated solution.
Overall, the progression moves from sequential reasoning (CoT) → branching evaluation (ToT) → structured synthesis (GoT).


| Aspect                   | Chain of Thought (CoT)         | Tree of Thought (ToT)                       | Graph of Thought (GoT)                            |
| ------------------------ | ------------------------------ | ------------------------------------------- | ------------------------------------------------- |
| **Thinking Style**       | Linear, step-by-step reasoning | Branching exploration of multiple ideas     | Networked reasoning with synthesis                |
| **Reasoning Flow**       | Sequential logical steps       | Parallel solution generation + evaluation   | Divergence → combination of insights              |
| **Main Goal**            | Improves reasoning accuracy     | Explores alternatives and picks the best solution | Integrates multiple perspectives                   |
| **Best For**             | Math problems, logic tasks     | Decision making, strategy generation        | Complex synthesis, multidisciplinary problems     |
| **Model Behavior**       | Explains reasoning explicitly  | Generates options then judges them          | Combines diverse outputs into one answer          |
| **Prompting Style Used** | Step-by-step reasoning prompt  | Branch generation + judge prompt            | Divergence prompt + aggregator prompt             |
| **Weakness Addressed**   | Reduces shallow guessing       | Avoids single-path bias                     | Prevents fragmented ideas; encourages integration |
| **Complexity Level**     | Low–Moderate                   | Moderate                                    | High                                              |
| **Output Type**          | Sequential explanation         | Evaluated best solution                     | Synthesized comprehensive solution                |
| **LCEL Pattern Used**    | Simple sequential chain        | RunnableParallel branches → judge chain     | RunnableParallel drafts → aggregation chain       |
